# FrugalOptiX on RFMiD: full run
**Setup (right panel):** Accelerator = GPU (T4 or P100) · Internet = On.
**Inputs:** (1) the RFMiD dataset, (2) your dataset that contains `FrugalOptiX_v4.tar.gz` (or the extracted `FrugalOptiX` folder).
**Run:** *Save Version → Save & Run All (Commit)* so it keeps running if you close the tab. About 4-5 hours.
Afterwards download `FrugalOptiX_results.zip` from the saved version's **Output** tab.

In [1]:
# 1. Unpack the project (works whether Kaggle left the .tar.gz packed or already extracted it)
import glob, os, shutil, subprocess
tars = glob.glob('/kaggle/input/**/FrugalOptiX_v4.tar.gz', recursive=True) + glob.glob('/kaggle/input/**/*.tgz', recursive=True)
if tars:
    subprocess.run(['tar', '-xzf', tars[0], '-C', '/kaggle/working'], check=True)
else:
    hits = [os.path.dirname(os.path.dirname(p)) for p in glob.glob('/kaggle/input/**/scripts/run_all.sh', recursive=True)]
    assert hits, 'Project not found. Add the dataset that contains FrugalOptiX_v4.tar.gz as an input.'
    shutil.copytree(hits[0], '/kaggle/working/FrugalOptiX', dirs_exist_ok=True)
os.chdir('/kaggle/working/FrugalOptiX')
print(os.getcwd(), sorted(os.listdir('.')))

/kaggle/working/FrugalOptiX ['.git', '.github', '.gitignore', 'README.md', 'RUNNING.md', 'configs', 'data', 'docs', 'frugaloptix', 'kaggle_run.ipynb', 'notebooks', 'requirements.txt', 'results', 'runs', 'scripts', 'test_photos', 'tests', 'web']


In [2]:
# 2. Install what Kaggle does not have, and check the GPU
!pip install -q optiland==0.6.3 onnx onnxruntime scikit-image PyYAML
import torch
print('GPU available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Turn on a GPU accelerator in Session options, then restart'

GPU available: True


In [3]:
# 3. Find the RFMiD folder and merge it into one labelled image folder (images shrunk to 768 px)
import glob, os
labels = [p for p in glob.glob('/kaggle/input/**/*[Ll]abel*.csv', recursive=True) if 'FrugalOptiX' not in p]
print('label files:', labels)
assert labels, 'RFMiD label CSVs not found. Add the RFMiD dataset as an input.'
root = os.path.commonpath([os.path.dirname(p) for p in labels])
if os.path.basename(root).lower() in ('training_set', 'validation_set', 'test_set', 'evaluation_set', '1. original images'):
    root = os.path.dirname(root)
print('root =', root)
!python scripts/merge_rfmid.py --root "{root}" --out data/RFMiD --max-side 768

label files: ['/kaggle/input/datasets/ozlemhakdagli/retinal-fundus-multi-disease-image-dataset-rfmid/Test_set/RFMiD_Testing_Labels.csv', '/kaggle/input/datasets/ozlemhakdagli/retinal-fundus-multi-disease-image-dataset-rfmid/Training_set/RFMiD_Training_Labels.csv', '/kaggle/input/datasets/ozlemhakdagli/retinal-fundus-multi-disease-image-dataset-rfmid/Validation_set/RFMiD_Validation_Labels.csv']
root = /kaggle/input/datasets/ozlemhakdagli/retinal-fundus-multi-disease-image-dataset-rfmid
3200 labelled images, 2531 abnormal, 0 labels without an image file -> data/RFMiD


In [4]:
# 4. THE RUN (live progress with minutes elapsed; also saved to /kaggle/working/run_log.txt)
import os, subprocess, sys, time
os.chdir('/kaggle/working/FrugalOptiX')

# fix: the restorer-study step must not receive --pretrained
p = 'scripts/run_all.sh'; s = open(p).read()
s = s.replace('03b_restorer_study.py --epochs $EP_R "$@"', '03b_restorer_study.py --epochs $EP_R --reuse')
s = s.replace('10_deployment_benchmark.py\n', '10_deployment_benchmark.py --size $SIZE\n')
open(p, 'w').write(s); print('patched:', '--reuse' in s)

# SEEDS="0 1 2" for the final numbers (about 1-2 h more per extra seed). EP_R = restorer epochs.
env = dict(os.environ, SEEDS="0", PYTHONUNBUFFERED="1", EP_R="20")
t0 = time.time()
pr = subprocess.Popen('bash scripts/run_all.sh rfmid data/RFMiD --pretrained', shell=True, env=env,
                      stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
with open('/kaggle/working/run_log.txt', 'w') as log:
    for line in pr.stdout:
        if 'warning' in line.lower(): continue
        m = f"[{int((time.time()-t0)//60):3d} min] {line.rstrip()}"
        log.write(m + '\n'); log.flush(); print(m); sys.stdout.flush()
pr.wait(); print('exit code', pr.returncode, '| total', round((time.time()-t0)/60), 'min')
assert pr.returncode == 0, 'run failed, see run_log.txt'

patched: True
[  0 min] {'train': 2240, 'val': 480, 'test': 480} files hashed: 1920
[  0 min] field weights used for the polymer design: [1.4575324996321788, 1.0108301913408955, 0.5316373090269255] (from saliency)
[  2 min]   PMMA+Polycarbonate: spots [18.8, 19.6, 56.8] mass 0.35 g  (120s)
[  3 min]   PMMA+PMMA: spots [16.0, 28.3, 59.6] mass 0.33 g  (211s)
[  5 min]   Zeonex+Polycarbonate: spots [16.5, 24.1, 75.7] mass 0.29 g  (335s)
[  7 min]   Zeonex+Zeonex: spots [17.3, 20.4, 62.6] mass 0.27 g  (460s)
[  9 min]   PMMA+Zeonex: spots [20.6, 40.9, 74.6] mass 0.30 g  (584s)
[ 11 min]   3-element fallback spots [35.8, 42.3, 84.8] DISCARDED (did not improve on 2 elements: optimiser did not converge)
[ 11 min]   X-ray done (671s)
[ 12 min]   yield vs floor multiple (rows) at grade scale 1x: {1.5: (0.0, 1.0), 2: (0.0, 1.0), 3: (0.7, 1.0), 4: (1.0, 1.0), 6: (1.0, 1.0)} (polymer, glass)
[ 12 min] main polymer design: PMMA+Polycarbonate spots [18.8, 19.6, 56.8] (glass [10.6, 9.1, 19.3]), mass 

In [5]:
# 5. Check the results are real, then package everything you need
import json, os, zipfile
os.chdir('/kaggle/working')
meta = json.load(open('FrugalOptiX/results/results.json'))['meta']
print(meta['dataset'], '| test images', meta['n_test'], '| synthetic:', meta['synthetic'])
assert not meta['synthetic'], 'results are still synthetic'
with zipfile.ZipFile('FrugalOptiX_results.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for top in ['FrugalOptiX/results', 'FrugalOptiX/web']:
        for r, _, fs in os.walk(top):
            for f in fs: z.write(os.path.join(r, f))
    for f in ['FrugalOptiX/runs/restorer_selected.pt', 'FrugalOptiX/runs/restorer.onnx']:
        z.write(f)          # raises an error if the file is missing
print(round(os.path.getsize('FrugalOptiX_results.zip') / 1e6, 1), 'MB  ->  download FrugalOptiX_results.zip from the Output tab')

RFMiD | test images 480 | synthetic: False
4.6 MB  ->  download FrugalOptiX_results.zip from the Output tab
